In [ ]:
import os
import numpy as np
import librosa
import soundfile as sf
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.signal import butter, sosfilt
from tqdm import tqdm

# ----------------------------
# Paths
# ----------------------------
clean_folder = r"C:\Users\benji\Documents\Bird_Project_Code\Filter_Bluetit\Bluetit"
noisy_folder = r"C:\Users\benji\Documents\Bird_Project_Code\Filter_Bluetit\Stream_Bluetit_Combined"
output_folder = r"C:\Users\benji\Documents\Bird_Project_Code\Filter_Bluetit\Denoised_Bluetit_FNN_Filter"
os.makedirs(output_folder, exist_ok=True)

# ----------------------------
# Functions
# ----------------------------
# def high_pass_filter(audio, sr, cutoff=500):
#     sos = butter(10, cutoff, btype='highpass', fs=sr, output='sos')
#     return sosfilt(sos, audio)

# ----------------------------
# Load and Process Audio Files
# ----------------------------
clean_files = sorted([f for f in os.listdir(clean_folder) if f.endswith(".wav")])
noisy_files = sorted([f for f in os.listdir(noisy_folder) if f.endswith(".wav")])

assert len(clean_files) == len(noisy_files), "Mismatch between clean and noisy files!"

all_clean_audio, all_noisy_audio, sample_rates = [], [], []
for clean_file, noisy_file in tqdm(zip(clean_files, noisy_files), total=len(clean_files)):
    clean_path, noisy_path = os.path.join(clean_folder, clean_file), os.path.join(noisy_folder, noisy_file)
    clean_audio, sr = librosa.load(clean_path, sr=None)
    noisy_audio, sr_noisy = librosa.load(noisy_path, sr=None)

    min_len = min(len(clean_audio), len(noisy_audio))
    clean_audio, noisy_audio = clean_audio[:min_len], noisy_audio[:min_len]

    # clean_audio, noisy_audio = high_pass_filter(librosa.util.normalize(clean_audio), sr), high_pass_filter(librosa.util.normalize(noisy_audio), sr)
    
    all_clean_audio.append(clean_audio)
    all_noisy_audio.append(noisy_audio)
    sample_rates.append(sr)

# ----------------------------
# Convert to STFT
# ----------------------------
n_fft, hop_length = 1024, 512
all_clean_stft = [librosa.stft(audio, n_fft=n_fft, hop_length=hop_length) for audio in all_clean_audio]
all_noisy_stft = [librosa.stft(audio, n_fft=n_fft, hop_length=hop_length) for audio in all_noisy_audio]

all_clean_mag, all_noisy_mag = [np.abs(stft) for stft in all_clean_stft], [np.abs(stft) for stft in all_noisy_stft]
clean_mag_dataset, noisy_mag_dataset = np.concatenate(all_clean_mag, axis=1), np.concatenate(all_noisy_mag, axis=1)

# ----------------------------
# Create Dataset
# ----------------------------
class DenoisingDataset(Dataset):
    def __init__(self, noisy_mag, clean_mag):
        self.inputs, self.targets = torch.tensor(noisy_mag.T, dtype=torch.float32), torch.tensor(clean_mag.T, dtype=torch.float32)
    def __len__(self): return self.inputs.shape[0]
    def __getitem__(self, idx): return self.inputs[idx], self.targets[idx]

dataset, dataloader = DenoisingDataset(noisy_mag_dataset, clean_mag_dataset), DataLoader(DenoisingDataset(noisy_mag_dataset, clean_mag_dataset), batch_size=32, shuffle=True)

# ----------------------------
# Define Enhanced Neural Network
# ----------------------------
class EnhancedDenoiser(nn.Module):
    def __init__(self, input_dim):
        super(EnhancedDenoiser, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024), nn.ReLU(), nn.BatchNorm1d(1024),
            nn.Linear(1024, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256),
            nn.Linear(256, input_dim), nn.ReLU()
        )
    def forward(self, x): return self.net(x)

input_dim, device = n_fft // 2 + 1, torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EnhancedDenoiser(input_dim).to(device)

# ----------------------------
# Train the Model
# ----------------------------
criterion, optimizer = nn.MSELoss(), optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler, num_epochs = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5), 300

print("Starting training...")
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad(), criterion(model(inputs), targets).backward(), optimizer.step()
        epoch_loss += criterion(model(inputs), targets).item() * inputs.size(0)
    
    scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(dataset):.6f}")

# ----------------------------
# Denoise Each Audio and Save

# ----------------------------
model.eval()
for i, (noisy_mag, noisy_stft, sr, filename) in enumerate(zip(all_noisy_mag, all_noisy_stft, sample_rates, noisy_files)):
    with torch.no_grad():
        denoised_frames = model(torch.tensor(noisy_mag.T, dtype=torch.float32).to(device)).cpu().numpy().T

    # phase, denoised_stft = np.angle(noisy_stft), denoised_frames * np.exp(1j * phase)
    phase = np.angle(noisy_stft)
    denoised_stft = denoised_frames * np.exp(1j * phase)

    denoised_audio = librosa.istft(denoised_stft, hop_length=hop_length)

    output_path = os.path.join(output_folder, f"denoised_{filename}")
    sf.write(output_path, denoised_audio, sr)
    print(f"Denoised audio saved to: {output_path}")

torch.save(model.state_dict(), r"C:\Users\benji\Documents\Bird_Project_Code\Filter_Bluetit\StreamBluetit_FNN_Filter_Trained.pth")


100%|██████████| 13/13 [00:00<00:00, 31.57it/s]


Starting training...
Epoch 1/300, Loss: 1.865294
Epoch 2/300, Loss: 1.734673
Epoch 3/300, Loss: 1.657659
Epoch 4/300, Loss: 1.593301
Epoch 5/300, Loss: 1.540120
Epoch 6/300, Loss: 1.484213
Epoch 7/300, Loss: 1.433638
Epoch 8/300, Loss: 1.392216
Epoch 9/300, Loss: 1.352777
Epoch 10/300, Loss: 1.330319
Epoch 11/300, Loss: 1.285415
Epoch 12/300, Loss: 1.248293
Epoch 13/300, Loss: 1.209396
Epoch 14/300, Loss: 1.181730
Epoch 15/300, Loss: 1.166215
Epoch 16/300, Loss: 1.142178
Epoch 17/300, Loss: 1.117983
Epoch 18/300, Loss: 1.088819
Epoch 19/300, Loss: 1.069979
Epoch 20/300, Loss: 1.058844
Epoch 21/300, Loss: 1.040517
Epoch 22/300, Loss: 1.024964
Epoch 23/300, Loss: 1.004912
Epoch 24/300, Loss: 0.985624
Epoch 25/300, Loss: 0.982698
Epoch 26/300, Loss: 0.966323
Epoch 27/300, Loss: 0.952658
Epoch 28/300, Loss: 0.946810
Epoch 29/300, Loss: 0.933223
Epoch 30/300, Loss: 0.924070
Epoch 31/300, Loss: 0.901241
Epoch 32/300, Loss: 0.887662
Epoch 33/300, Loss: 0.882694
Epoch 34/300, Loss: 0.876493
Ep